# W31 · OpenUSD 基础：场景图与组合

> 阶段四（选修前沿）第 1 周。OpenUSD 是 Isaac Sim 乃至整个 Omniverse 生态的「场景描述语言」，
> 理解它，就看懂了 Isaac Lab 中机器人、传感器、地形是怎么被组织起来的。

## 学习目标

学完本 notebook，你应该能够：

1. 说出 USD 的五个核心概念：**Stage、Layer、Prim、Property（Attribute/Relationship）、Schema**，并用一句话解释各自的角色；
2. 直接阅读 `.usda` 文本格式的场景文件，看懂 prim 层级、属性与变换；
3. 用 `pxr` Python API 创建、查询、遍历一个 USD Stage（需 `uv pip install usd-core`）；
4. 解释 USD 的**组合（Composition）**机制——Sublayers / References / Variants / Inherits——并说明它为什么适合组织「基础场景 + 域随机化覆盖层」。

## 运行前提

- **第一部分（USDA 文本格式）**：纯文本读写，任何环境都能运行 ✅
- **第二部分（pxr API）**：需要 `uv pip install usd-core`（纯 CPU 即可，不需要 GPU）。
  本机未安装 pxr 的 cell **保持未执行**，安装后从头运行即可。

## 1. 直觉：USD 是「3D 世界的 HTML + CSS」

写网页时，HTML 描述**结构**（标签树），CSS 描述**样式**，而且可以一层叠一层地覆盖。
USD 之于 3D 场景，正如 HTML+CSS 之于网页：

- 一个 USD **Stage（舞台）** = 整个 3D 世界（机器人 + 地形 + 灯光 + 物理参数）；
- Stage 由一层或多层 **Layer（图层）** 组合而成，上层可以**覆盖**下层的值——这就是 CSS 的层叠；
- 世界里的每个「东西」（机器人本体、一个关节、一盏灯、一个相机）是一个 **Prim**；
- Prim 身上的数据叫 **Property**，分为**属性（Attribute，存数值）**和**关系（Relationship，指向别的 Prim）**；
- Prim 的「类型」由 **Schema** 定义——`Xform` 表示可变换的物体，`Mesh` 表示网格，`PhysicsRevoluteJoint` 表示旋转关节。

为什么机器人仿真选中 USD？因为它解决了三个真问题：

1. **组合性**：实验室的机械臂、传送带、相机分别由不同人建模成 `.usd` 文件，像乐高一样拼进一个场景，互不修改源文件；
2. **可覆盖**：域随机化 = 在基础场景之上叠一个 Layer，把摩擦系数、质量改掉，基础场景原封不动；
3. **非破坏性编辑**：所有修改都以 Layer 的形式记录，随时可以把某一层关掉「还原现场」。

> 💡 类比：如果你用过 Git，Layer 之于场景就像 commit 之于代码——每层都是增量，最终 Stage 是所有层「合并」的结果。

## 2. 核心概念速查表

| 概念 | 一句话解释 | HTML 类比 |
|------|-----------|-----------|
| **Stage** | 组合完成的完整场景，`pxr.Usd.Stage` 是 API 入口 | 浏览器渲染出的最终页面 |
| **Layer** | 一个 `.usd`/`.usda` 文件，记录一批「意见（opinion）」 | 一个 CSS 文件 |
| **Prim** | 场景树中的一个节点，有路径如 `/World/Robot/base_link` | 一个 HTML 元素 |
| **Attribute** | Prim 上的键值数据，带类型，如 `double xformOp:translate` | 元素的 attribute |
| **Relationship** | 指向其他 Prim 的引用，如关节的 `body0 → /World/Robot/link1` | `<a href>` / 父子关系 |
| **Schema** | Prim 的类型系统（`def Xform`、`def Mesh`…） | 标签语义（`<div>` vs `<img>`） |

## 3. USDA：可以直接阅读的文本格式

USD 文件有三种后缀：

- `.usda` —— **ASCII 文本**，可以直接用编辑器打开，适合学习与版本管理（diff 友好）；
- `.usdc` —— 二进制（crate）格式，体积小、加载快，适合生产；
- `.usd` —— 容器后缀，既可以是文本也可以是二进制。

学习阶段一律用 `.usda`。下面这个 cell 纯用 Python 字符串写出一个「简化四旋翼」场景——
**不需要安装任何 USD 库**，因为 `.usda` 本质上就是约定格式的文本。

In [ ]:
from pathlib import Path

# 一个简化的四旋翼场景：机身 + 4 个旋翼，用 Xform 层级表达
usda_text = """#usda 1.0
(
    defaultPrim = "Quadrotor"
    upAxis = "Z"
    metersPerUnit = 1.0
    doc = "教学用简化四旋翼：机身 + 4 旋翼"
)

def Xform "Quadrotor"
{
    double3 xformOp:translate = (0, 0, 1.0)
    uniform token[] xformOpOrder = ["xformOp:translate"]

    def Xform "Body"
    {
        custom double mass = 0.027
        color3f primvars:displayColor = [(0.1, 0.4, 0.9)]
    }

    def Xform "Rotor_FL" { double3 xformOp:translate = ( 0.046,  0.046, 0.0)
                           uniform token[] xformOpOrder = ["xformOp:translate"] }
    def Xform "Rotor_FR" { double3 xformOp:translate = ( 0.046, -0.046, 0.0)
                           uniform token[] xformOpOrder = ["xformOp:translate"] }
    def Xform "Rotor_BL" { double3 xformOp:translate = (-0.046,  0.046, 0.0)
                           uniform token[] xformOpOrder = ["xformOp:translate"] }
    def Xform "Rotor_BR" { double3 xformOp:translate = (-0.046, -0.046, 0.0)
                           uniform token[] xformOpOrder = ["xformOp:translate"] }
}
"""

out_dir = Path("runs/usd_demos")
out_dir.mkdir(parents=True, exist_ok=True)
usda_path = out_dir / "quadrotor.usda"
usda_path.write_text(usda_text, encoding="utf-8")

print(f"已写入 {usda_path}（{usda_path.stat().st_size} 字节）")
print("-" * 50)
print(usda_path.read_text(encoding="utf-8"))

### 逐行解读

- `#usda 1.0` —— 文件头，**必须是第一行**；
- 顶部的 `(...)` 是 **Layer 元数据**：
  - `defaultPrim` —— 别的场景 reference 本文件时默认挂到哪个 Prim；
  - `upAxis = "Z"` —— Z 轴向上（机器人惯例；图形学界常用 Y 向上，导入模型时注意）；
  - `metersPerUnit = 1.0` —— 长度单位是米。**它只是一个声明**，USD 不会帮你换算单位；
- `def Xform "Quadrotor"` —— 定义一个类型为 `Xform`（带变换的组）的 Prim，花括号里是它的孩子；
- `double3 xformOp:translate` —— 一个 **Attribute**：类型 `double3`，名字 `xformOp:translate`。
  `xformOp:*` 是变换操作的命名空间，`xformOpOrder` 声明这些操作的**应用顺序**（先旋转还是先平移，结果完全不同——
  这正是控制课里「变换矩阵不满足交换律」的 USD 表达）；
- `custom double mass` —— `custom` 前缀表示这是**用户自定义属性**，不属于标准 Schema。
  Isaac Sim 大量使用 custom 属性记录物理/语义信息；
- Prim 的**路径**由层级决定：`/Quadrotor/Rotor_FL` —— 和 ROS 的 TF 树、文件系统路径是同一个思想。

In [ ]:
# .usda 是纯文本，不用 pxr 也能做「静态分析」。
# 统计场景里有多少个 Prim、每个 Prim 的类型和深度：
import re
from pathlib import Path

text = Path("runs/usd_demos/quadrotor.usda").read_text(encoding="utf-8")

print(f"{'深度':<4} {'类型':<10} 名称")
for line in text.splitlines():
    m = re.match(r'^(\s*)def\s+(\w+)\s+"([^"]+)"', line)
    if m:
        indent, prim_type, prim_name = m.groups()
        depth = len(indent) // 4  # 每级缩进 4 空格
        print(f"{depth:<6} {prim_type:<10} {prim_name}")

## 4. pxr：USD 的官方 Python API

文本格式能看懂结构，但要**程序化地**构建/查询/修改场景，就要用官方 API 了：

```bash
uv pip install usd-core   # 纯 Python 轮子，CPU 即可；装好后下面两个 cell 才能运行
```

> ⚠️ 以下 cell 依赖 `pxr`，本机未安装时**保持未执行**。
> 安装 usd-core 后从「Run All」重新运行本 notebook。

In [ ]:
# 用 pxr 从零创建一个 Stage（等价于上一节手写的文本，但完全程序化）
from pxr import Usd, UsdGeom, Gf, Sdf

stage = Usd.Stage.CreateNew("runs/usd_demos/quadrotor_pxr.usda")

# Layer 元数据
UsdGeom.SetStageUpAxis(stage, UsdGeom.Tokens.z)
UsdGeom.SetStageMetersPerUnit(stage, 1.0)

# 定义 Prim 层级：/Quadrotor/Body
quad = UsdGeom.Xform.Define(stage, "/Quadrotor")
stage.SetDefaultPrim(quad.GetPrim())
quad.AddTranslateOp().Set(Gf.Vec3d(0, 0, 1.0))

body = UsdGeom.Xform.Define(stage, "/Quadrotor/Body")
# 自定义属性：custom 前缀在文本格式里，API 里用 SetCustomDataByKey 或直接 CreateAttribute
body.GetPrim().CreateAttribute("mass", Sdf.ValueTypeNames.Double, custom=True).Set(0.027)

# 批量创建 4 个旋翼
for name, (x, y) in {
    "Rotor_FL": (0.046, 0.046), "Rotor_FR": (0.046, -0.046),
    "Rotor_BL": (-0.046, 0.046), "Rotor_BR": (-0.046, -0.046),
}.items():
    rotor = UsdGeom.Xform.Define(stage, f"/Quadrotor/{name}")
    rotor.AddTranslateOp().Set(Gf.Vec3d(x, y, 0.0))

stage.GetRootLayer().Save()
print("已保存 runs/usd_demos/quadrotor_pxr.usda")

In [ ]:
# 遍历与查询：RL 代码里最常用的操作
from pxr import Usd

stage = Usd.Stage.Open("runs/usd_demos/quadrotor_pxr.usda")

# 1) 深度优先遍历整棵 Prim 树
for prim in stage.Traverse():
    print(f"{str(prim.GetPath()):<28} type={prim.GetTypeName()}")

# 2) 按路径取 Prim，再读 Attribute
body = stage.GetPrimAtPath("/Quadrotor/Body")
mass = body.GetAttribute("mass").Get()
print(f"\n/Quadrotor/Body 的 mass = {mass}")

# 3) 时间采样：Attribute 的值可以随时间变化（动画/轨迹），time=Usd.TimeCode.Default() 取默认值
translate = stage.GetPrimAtPath("/Quadrotor").GetAttribute("xformOp:translate")
print(f"/Quadrotor 的平移 = {translate.Get()}")

## 5. 组合（Composition）：USD 的灵魂

如果说 Prim 树是 USD 的「骨架」，**组合机制**就是它的「基因重组技术」。
一个 Stage 从不直接等于某个文件，而是多条 Layer **按强度顺序合并（compose）** 的结果。
四条最常用的组合弧（composition arcs）：

| 组合弧 | 作用 | 编程类比 | 机器人场景中的用途 |
|--------|------|----------|--------------------|
| **Sublayers（子层）** | 一个 Layer 叠在另一个之上，上层值覆盖下层 | CSS 层叠 | 基础场景 + 域随机化覆盖层 |
| **References（引用）** | 把外部文件的整棵 Prim 树「挂」到指定路径 | `#include` / 模块导入 | 同一机器人模型实例化 4096 份 |
| **Variants（变体）** | 一个 Prim 内置多套可选配置，运行时切换 | 策略模式 / 配置档位 | 同一机械臂切换不同夹爪 |
| **Inherits（继承）** | 一个 Prim 继承另一个 Prim 的全部属性 | 类继承 | 多款机器人共享公共参数 |

合并时谁说了算？USD 用助记词 **LIVRPS** 表示意见强度从强到弱：
**L**ocal（本层直接写的值）> **I**nherits > **V**ariants > **R**eferences > **P**ayloads（延迟加载的引用）> **S**pecializes。

**关键直觉**：Sublayer 的覆盖是*非破坏*的——下层文件一个字节都不会改，
就像你在照片编辑软件里加了一个调整图层。这正是「域随机化不污染基础资产」的工程实现。

In [ ]:
# 组合实战：base.usda + dr.usda 两层叠加，上层覆盖摩擦系数
from pxr import Usd, Sdf

# --- 第 1 层：基础场景（资产作者的原始文件，永不被修改） ---
base = """#usda 1.0
(
    defaultPrim = "World"
)
def Xform "World"
{
    def Xform "Ground"
    {
        custom double friction = 0.8
    }
}
"""
with open("runs/usd_demos/base.usda", "w") as f:
    f.write(base)

# --- 第 2 层：域随机化覆盖层（RL 训练脚本每次 rollout 前生成） ---
dr = """#usda 1.0
(
    defaultPrim = "World"
    subLayers = [@./base.usda@]
)
# over（而非 def）表示：只覆盖已存在 Prim 的意见，不创建新 Prim
over "World"
{
    over "Ground"
    {
        custom double friction = 0.3   # 本 episode 随机到的低摩擦地面
    }
}
"""
with open("runs/usd_demos/dr.usda", "w") as f:
    f.write(dr)

# 打开 dr.usda：Stage 自动把 subLayers 合并，/World/Ground 的 friction 读到 0.3
stage = Usd.Stage.Open("runs/usd_demos/dr.usda")
ground = stage.GetPrimAtPath("/World/Ground")
print("组合后 friction =", ground.GetAttribute("friction").Get())  # 0.3（上层获胜）

# 而基础文件原封不动：
stage_base = Usd.Stage.Open("runs/usd_demos/base.usda")
print("base.usda 中 friction 仍为 =",
      stage_base.GetPrimAtPath("/World/Ground").GetAttribute("friction").Get())  # 0.8

## 6. USD 与 Isaac Sim 的关系 & 本周小结

- **Isaac Sim 的场景就是一个 USD Stage**：你用 Isaac Lab 配置写的每一个资产、每个传感器，
  最终都被 spawn 成 Stage 上的 Prim；`{ENV_REGEX_NS}`（如 `/World/envs/env_.*/Robot`）
  就是 USD 的路径通配——4096 个并行环境 = 4096 份 reference 进 Stage 的机器人 Prim 树；
- PhysX 的物理参数（质量、摩擦、关节驱动）也以 USD Attribute（Physics Schema）的形式存在，
  域随机化在 Isaac Lab 里走的正是「改 Attribute / 叠 Layer」这条路；
- 学会读 `.usda`，你就能直接打开任何 Isaac Sim 资产，搞清楚它的 Prim 结构与属性——
  这是调试仿真问题的基本功。

**本周要点回顾**：Stage = Layer 组合的结果；Prim 树组织万物；Attribute 存数据；
LIVRPS 决定覆盖优先级；组合是非破坏性的。

---

## ✏️ 练习

> 完成后把交付物放入 `runs/usd_demos/` 或你的 `journal/`。

1. **手写机械臂场景**（★，约 20 分钟）
   不用任何 USD 库，纯文本写一个 `arm_scene.usda`：根 Prim `/Arm` 下依次嵌套
   `Base → Link1 → Link2` 三个 `Xform`，每级沿 Z 轴偏移 0.1 m，给 `Link2` 加自定义属性
   `custom double payload = 0.5`。**交付物**：`arm_scene.usda` 文件 + 用本 notebook 第 3 节的
   正则脚本打印出的 Prim 树截图/输出。
2. **单位陷阱**（★，约 15 分钟）
   某同学把 `quadrotor.usda` 的 `metersPerUnit` 从 `1.0` 改成 `0.01`（想用厘米），
   然后在 Isaac Sim 里发现旋翼位置全错了。解释原因，并给出正确做法。
   **交付物**：不超过 150 字的书面解释。
3. **pxr 编程读写**（★★，约 30 分钟，需 `uv pip install usd-core`）
   写脚本读取 `runs/usd_demos/quadrotor_pxr.usda`，打印每个 Prim 的路径与类型；
   然后给 `/Quadrotor/Body` 添加 `custom double battery_voltage = 7.4` 并保存。
   **交付物**：`journal/usd_query.py` 脚本。
4. **设计域随机化 Layer 结构**（★★，约 25 分钟）
   为「四足机器人过草地」任务设计三层 Layer：`base.usda`（地形+机器人）、
   `terrain_dr.usda`（摩擦/起伏随机化）、`robot_dr.usda`（质量/电机强度随机化）。
   用文字说明三层的 subLayers 叠放顺序与理由，并回答：如果 `base.usda` 里写了
   `friction = 0.8` 而 `terrain_dr.usda` 写了 `0.3`，最终生效的是哪个？为什么？
   **交付物**：`journal/layer_design.md`。

## 参考答案

<details>
<summary>练习 1：手写机械臂场景（参考答案）</summary>

```usda
#usda 1.0
(
    defaultPrim = "Arm"
    upAxis = "Z"
    metersPerUnit = 1.0
)

def Xform "Arm"
{
    def Xform "Base"
    {
        double3 xformOp:translate = (0, 0, 0.1)
        uniform token[] xformOpOrder = ["xformOp:translate"]

        def Xform "Link1"
        {
            double3 xformOp:translate = (0, 0, 0.1)
            uniform token[] xformOpOrder = ["xformOp:translate"]

            def Xform "Link2"
            {
                double3 xformOp:translate = (0, 0, 0.1)
                uniform token[] xformOpOrder = ["xformOp:translate"]
                custom double payload = 0.5
            }
        }
    }
}
```

要点：`def` 创建 Prim；层级 = 嵌套花括号；每级 `xformOp:translate` 是相对于**父 Prim** 的
局部变换（正运动学的连乘正是这样发生的）；`custom` 前缀声明非标准属性。
评分自检：正则脚本应打印出 4 个 Prim（Arm/Base/Link1/Link2），深度 0–3。
</details>

<details>
<summary>练习 2：单位陷阱（参考答案）</summary>

`metersPerUnit` 是**元数据声明**，不是换算开关——USD 不会把文件里的数值乘以 0.01。
旋翼坐标 `(0.046, 0.046, 0)` 等仍以原数值存储，只是把「这些数字的单位」从米改成了厘米，
于是 Isaac Sim 按厘米解读 `(0.046, ...)` 得到的是 0.46 mm 的旋翼臂，自然全错。
**正确做法**：改 `metersPerUnit` 的同时，把所有长度数值（translate、尺寸、半径）手动换算；
更推荐始终用 `metersPerUnit = 1.0`（SI 单位），导入外部资产时用 `xformOp:scale` 做整体缩放。
</details>

<details>
<summary>练习 3：pxr 编程读写（参考答案）</summary>

```python
from pxr import Usd, Sdf

stage = Usd.Stage.Open("runs/usd_demos/quadrotor_pxr.usda")

for prim in stage.Traverse():
    print(f"{str(prim.GetPath()):<28} type={prim.GetTypeName()}")

body = stage.GetPrimAtPath("/Quadrotor/Body")
body.CreateAttribute("battery_voltage", Sdf.ValueTypeNames.Double, custom=True).Set(7.4)
stage.GetRootLayer().Save()

# 验证
print("battery_voltage =", body.GetAttribute("battery_voltage").Get())
```

易错点：忘记 `Save()`；`GetPrimAtPath` 拿到 `None` 时先检查路径字符串是否以 `/` 开头。
</details>

<details>
<summary>练习 4：域随机化 Layer 结构（参考答案）</summary>

推荐叠放顺序（**最上层最强**）：

```
robot_dr.usda        ← 最上层：机器人自身参数（质量、电机强度）
terrain_dr.usda      ← 中层：地形参数（摩擦、起伏）
base.usda            ← 底层：资产原始文件，永不被修改
```

即 `terrain_dr.usda` 的 `subLayers = [@./base.usda@]`，
`robot_dr.usda` 的 `subLayers = [@./terrain_dr.usda@]`（或直接并列 subLayers，先写的更强）。

理由：① base 保持只读，DR 全在上层以 `over` 覆盖，符合非破坏编辑；
② 机器人参数与地形参数分层，方便独立开关做消融（只随机地形 / 只随机机器人）。

若 base 写 `friction = 0.8`、terrain_dr 写 `0.3`：生效的是 **0.3**——
在 LIVRPS 强度规则下，更上层（更靠近 Stage 根 Layer）的 Local 意见更强。
</details>

---

## 延伸阅读

- [OpenUSD 官方网站与文档](https://openusd.org/)（概念、API、教程入口）
- [GitHub: PixarAnimationStudios/OpenUSD](https://github.com/PixarAnimationStudios/OpenUSD)（源码与示例）
- [pypi: usd-core](https://pypi.org/project/usd-core/)（纯 Python 轮子的安装说明）
- [Isaac Sim 文档](https://docs.isaacsim.omniverse.nvidia.com/)（USD 在 Isaac Sim 中的应用）
- [Isaac Lab 文档](https://isaac-sim.github.io/IsaacLab/)（下一周的主角，先浏览目录）